# 🐍 Module 4 — Serving and Production
**ml-engineering-drills**

| Section | Content |
|---|---|
| 4.1 | FastAPI — routing, Pydantic I/O, lifespan, error handling |
| 4.2 | Concurrency — async/await, ThreadPoolExecutor, asyncio.gather |
| 4.3 | Testing — pytest, fixtures, TestClient, mocking |


---
## 4.1 FastAPI

### Lesson — routing, Pydantic I/O, lifespan

In [ ]:
# All FastAPI apps in this module are tested via TestClient
# so every cell is fully executable in the notebook.

from fastapi import FastAPI, HTTPException, Depends, Query, Path
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field
from contextlib import asynccontextmanager
from typing import Optional, List
import numpy as np
import joblib, time, logging

logger = logging.getLogger("api")

# ── 1. Minimal app ────────────────────────────────────────────
app_minimal = FastAPI(title="Minimal ML API")

@app_minimal.get("/health")
def health():
    return {"status": "ok"}

@app_minimal.get("/items/{item_id}")
def get_item(item_id: int, verbose: bool = Query(False)):
    if item_id < 0:
        raise HTTPException(status_code=404, detail=f"Item {item_id} not found")
    return {"id": item_id, "verbose": verbose}

client = TestClient(app_minimal)
print(client.get("/health").json())
print(client.get("/items/42?verbose=true").json())
print(client.get("/items/-1").status_code)   # 404

In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field, field_validator
from typing import Literal, List
import numpy as np, time, logging

# ── 2. Pydantic I/O + lifespan ────────────────────────────────
class PredictionRequest(BaseModel):
    age: int = Field(ge=18, le=120)
    tenure_months: int = Field(ge=0, le=240)
    monthly_charges: float = Field(gt=0, le=500)
    contract_type: Literal["month-to-month", "one-year", "two-year"]
    nb_products: int = Field(ge=1, le=10, default=1)

    @field_validator("monthly_charges")
    @classmethod
    def round_charges(cls, v: float) -> float:
        return round(v, 2)

class PredictionResponse(BaseModel):
    churn_proba: float = Field(ge=0.0, le=1.0)
    prediction: bool
    model_version: str
    latency_ms: float

class BatchPredictionRequest(BaseModel):
    requests: List[PredictionRequest]

class BatchPredictionResponse(BaseModel):
    predictions: List[PredictionResponse]
    total_latency_ms: float

# Fake model for demo purposes
class FakeChurnModel:
    version = "v1.2.0"
    def predict_proba(self, features: list) -> float:
        # deterministic fake score based on input
        age, tenure, charges, nb_prod = features
        score = max(0.0, min(1.0, 0.5 - tenure * 0.01 + charges * 0.002))
        return round(score, 4)

# ── lifespan: load model ONCE at startup ──────────────────────
# NOTE: lifespan is incompatible with Jupyter's running event loop.
# Loading model directly instead — equivalent behavior.
ml = {
    "model":   FakeChurnModel(),
    "version": FakeChurnModel.version,
}

app = FastAPI(title="Churn Prediction API", version="1.0.0")

@app.get("/health")
def health():
    return {"status": "ok", "model_loaded": "model" in ml}

@app.get("/metrics")
def metrics():
    return {"model_version": ml.get("version"), "status": "serving"}

@app.post("/predict", response_model=PredictionResponse)
def predict(req: PredictionRequest):
    t0 = time.perf_counter()
    try:
        features = [req.age, req.tenure_months, req.monthly_charges, req.nb_products]
        proba    = ml["model"].predict_proba(features)
        return PredictionResponse(
            churn_proba    = proba,
            prediction     = proba > 0.5,
            model_version  = ml["version"],
            latency_ms     = round((time.perf_counter() - t0) * 1000, 2),
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/predict/batch", response_model=BatchPredictionResponse)
def predict_batch(batch: BatchPredictionRequest):
    t0 = time.perf_counter()
    preds = []
    for req in batch.requests:
        features = [req.age, req.tenure_months, req.monthly_charges, req.nb_products]
        proba    = ml["model"].predict_proba(features)
        preds.append(PredictionResponse(
            churn_proba   = proba,
            prediction    = proba > 0.5,
            model_version = ml["version"],
            latency_ms    = 0.0,
        ))
    return BatchPredictionResponse(
        predictions      = preds,
        total_latency_ms = round((time.perf_counter() - t0) * 1000, 2),
    )

# ── Tests via TestClient ──────────────────────────────────────
client = TestClient(app)

print("\n=== /health ===")
print(client.get("/health").json())

print("\n=== /predict (valid) ===")
r = client.post("/predict", json={
    "age": 35, "tenure_months": 24,
    "monthly_charges": 75.5, "contract_type": "one-year",
})
print(r.json())

print("\n=== /predict (invalid age) ===")
r = client.post("/predict", json={
    "age": 150, "tenure_months": 24,
    "monthly_charges": 75.5, "contract_type": "one-year",
})
print(r.status_code, r.json()["detail"][0]["msg"])

print("\n=== /predict/batch ===")
r = client.post("/predict/batch", json={"requests": [
    {"age": 30, "tenure_months": 12, "monthly_charges": 60.0, "contract_type": "month-to-month"},
    {"age": 55, "tenure_months": 48, "monthly_charges": 90.0, "contract_type": "two-year"},
]})
resp = r.json()
print(f"batch of {len(resp['predictions'])} | total {resp['total_latency_ms']}ms")

### Lesson — Depends: dependency injection

In [ ]:
from fastapi import FastAPI, HTTPException, Depends, Header
from fastapi.testclient import TestClient
from pydantic import BaseModel
from typing import Optional, Annotated

# ── Depends — the FastAPI way to share logic ──────────────────
# Use cases: auth, DB connections, config injection, rate limiting

# ── 1. API key auth ───────────────────────────────────────────
VALID_API_KEYS = {"secret-key-123", "dev-key-456"}

def verify_api_key(x_api_key: Annotated[Optional[str], Header()] = None):
    if x_api_key not in VALID_API_KEYS:
        raise HTTPException(status_code=401, detail="Invalid or missing API key")
    return x_api_key

# ── 2. Pagination params ──────────────────────────────────────
class Pagination:
    def __init__(self, skip: int = 0, limit: int = Query(default=10, le=100)):
        self.skip  = skip
        self.limit = limit

# ── 3. Shared model loader ────────────────────────────────────
class ModelStore:
    def __init__(self):
        self._model = {"version": "v1.0", "threshold": 0.5}

    def get(self) -> dict:
        return self._model

model_store = ModelStore()

def get_model(store: ModelStore = Depends(lambda: model_store)) -> dict:
    return store.get()

# ── App using dependencies ────────────────────────────────────
app_deps = FastAPI()

@app_deps.get("/secure-data")
def secure_data(api_key: str = Depends(verify_api_key)):
    return {"data": "top secret", "accessed_by_key": api_key[:8] + "..."}

@app_deps.get("/items")
def list_items(pagination: Pagination = Depends()):
    all_items = list(range(100))
    return {
        "items": all_items[pagination.skip:pagination.skip + pagination.limit],
        "skip":  pagination.skip,
        "limit": pagination.limit,
    }

@app_deps.get("/model-info")
def model_info(model: dict = Depends(get_model)):
    return model

# ── Tests ─────────────────────────────────────────────────────
client = TestClient(app_deps)

print("=== Auth: valid key ===")
r = client.get("/secure-data", headers={"x-api-key": "secret-key-123"})
print(r.json())

print("\n=== Auth: invalid key ===")
r = client.get("/secure-data", headers={"x-api-key": "wrong"})
print(r.status_code, r.json())

print("\n=== Pagination ===")
r = client.get("/items?skip=10&limit=5")
print(r.json())

print("\n=== Model info ===")
print(client.get("/model-info").json())

# KEY INSIGHT:
# Depends() → called once per request, result injected automatically
# Nested Depends → FastAPI resolves the full dependency graph
# Depends on a class → __init__ params become query params automatically

### 🏋️ Exercises 4.1

In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 1 — Full prediction API with versioning
# ════════════════════════════════════════════════════════
# Build a FastAPI app with:
# - GET  /health         → {"status": "ok", "models": [list of loaded versions]}
# - POST /v1/predict     → uses model "v1"
# - POST /v2/predict     → uses model "v2" (different threshold: 0.4)
# - GET  /models         → list all available model versions + their thresholds
#
# Both predict endpoints accept the same PredictionRequest schema
# and return PredictionResponse.
# Load both models in lifespan.
# Each model is just a dict: {"version": str, "threshold": float}
# Proba = fake: 0.3 + tenure_months * 0.01 (clipped to [0,1])
#
# Write 4 tests covering: health, v1 predict, v2 predict, model list.

from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field
from contextlib import asynccontextmanager
from typing import Literal, List
import time

class PredictionRequest(BaseModel):
    age: int = Field(ge=18, le=120)
    tenure_months: int = Field(ge=0)
    monthly_charges: float = Field(gt=0)
    contract_type: Literal["month-to-month", "one-year", "two-year"]

class PredictionResponse(BaseModel):
    churn_proba: float
    prediction: bool
    model_version: str
    threshold_used: float

# complete here
app_ex1 = None

if app_ex1:
    client = TestClient(app_ex1)
    # write your 4 tests here

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# models = {}
#
# @asynccontextmanager
# async def lifespan(app):
#     models["v1"] = {"version": "v1", "threshold": 0.5}
#     models["v2"] = {"version": "v2", "threshold": 0.4}
#     yield
#     models.clear()
#
# app_ex1 = FastAPI(lifespan=lifespan)
#
# def fake_proba(tenure: int) -> float:
#     return round(min(1.0, max(0.0, 0.3 + tenure * 0.01)), 4)
#
# def make_prediction(req, version):
#     model   = models[version]
#     proba   = fake_proba(req.tenure_months)
#     return PredictionResponse(
#         churn_proba   = proba,
#         prediction    = proba > model["threshold"],
#         model_version = model["version"],
#         threshold_used= model["threshold"],
#     )
#
# @app_ex1.get("/health")
# def health():
#     return {"status": "ok", "models": list(models.keys())}
#
# @app_ex1.post("/v1/predict", response_model=PredictionResponse)
# def predict_v1(req: PredictionRequest):
#     return make_prediction(req, "v1")
#
# @app_ex1.post("/v2/predict", response_model=PredictionResponse)
# def predict_v2(req: PredictionRequest):
#     return make_prediction(req, "v2")
#
# @app_ex1.get("/models")
# def list_models():
#     return [{"version": v, "threshold": m["threshold"]} for v, m in models.items()]
#
# # Tests
# client = TestClient(app_ex1)
# r = client.get("/health"); assert r.json()["models"] == ["v1","v2"]
# r = client.post("/v1/predict", json={"age":30,"tenure_months":20,
#     "monthly_charges":60,"contract_type":"one-year"})
# assert r.json()["threshold_used"] == 0.5
# r = client.post("/v2/predict", json={"age":30,"tenure_months":20,
#     "monthly_charges":60,"contract_type":"one-year"})
# assert r.json()["threshold_used"] == 0.4
# r = client.get("/models"); assert len(r.json()) == 2
# print("All tests passed")

In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 2 — Depends: rate limiter + request logger
# ════════════════════════════════════════════════════════
# Build a FastAPI app with two dependencies:
#
# a) RateLimiter(max_requests: int, window_seconds: float)
#    - tracks call count per window using a simple in-memory counter
#    - raises HTTP 429 if limit exceeded
#    - resets counter after window_seconds
#
# b) log_request (simple function dependency)
#    - logs "REQUEST {method} {path}" at INFO level
#    - returns None (side-effect only)
#
# Apply both dependencies to POST /predict.
# Write tests for: normal call, rate limit exceeded.

from fastapi import FastAPI, HTTPException, Depends, Request
from fastapi.testclient import TestClient
from pydantic import BaseModel
import time, logging, sys

logging.basicConfig(level=logging.INFO, stream=sys.stdout,
                    format="%(levelname)s | %(message)s")
logger = logging.getLogger("api")

class SimpleRequest(BaseModel):
    value: float

app_ex2 = FastAPI()

# complete here: RateLimiter class and log_request function

# @app_ex2.post("/predict")
# def predict(req: SimpleRequest,
#             _log = Depends(log_request),
#             _rl  = Depends(RateLimiter(max_requests=3, window_seconds=2.0))):
#     return {"result": req.value * 2}

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# class RateLimiter:
#     def __init__(self, max_requests: int, window_seconds: float):
#         self.max_requests    = max_requests
#         self.window_seconds  = window_seconds
#         self._count          = 0
#         self._window_start   = time.time()
#
#     def __call__(self):
#         now = time.time()
#         if now - self._window_start > self.window_seconds:
#             self._count = 0
#             self._window_start = now
#         self._count += 1
#         if self._count > self.max_requests:
#             raise HTTPException(status_code=429,
#                                 detail="Rate limit exceeded")
#
# def log_request(request: Request):
#     logger.info(f"REQUEST {request.method} {request.url.path}")
#
# rate_limiter = RateLimiter(max_requests=3, window_seconds=2.0)
#
# @app_ex2.post("/predict")
# def predict(req: SimpleRequest,
#             _log = Depends(log_request),
#             _rl  = Depends(rate_limiter)):
#     return {"result": req.value * 2}
#
# client = TestClient(app_ex2)
# payload = {"value": 5.0}
# for i in range(3):
#     r = client.post("/predict", json=payload)
#     assert r.status_code == 200, f"call {i+1} failed"
# r = client.post("/predict", json=payload)
# assert r.status_code == 429
# print("Rate limiter works correctly")

---
## 4.2 Concurrency

### Lesson — async/await, ThreadPoolExecutor, asyncio.gather

In [ ]:
import asyncio
import time
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
from typing import List

# ── I/O bound vs CPU bound ────────────────────────────────────
# I/O bound  → waiting for network, disk, DB queries
#              → async/await OR ThreadPoolExecutor
# CPU bound  → heavy computation (model training, feature engineering)
#              → ProcessPoolExecutor (bypasses GIL)
#              → NOT async (doesn't help for CPU)

# ── async/await: cooperative multitasking ────────────────────
async def fetch_data(source: str, delay: float) -> dict:
    "Simulates an async I/O call (network, DB query)"
    await asyncio.sleep(delay)   # non-blocking wait
    return {"source": source, "data": list(range(5))}

async def fetch_weather(region: str) -> dict:
    await asyncio.sleep(0.1)
    return {"region": region, "temp": 15.0}

async def fetch_consumption(region: str) -> dict:
    await asyncio.sleep(0.15)
    return {"region": region, "mw": 1200.0}

# Sequential vs parallel async
async def sequential():
    t0 = time.perf_counter()
    w  = await fetch_weather("IDF")
    c  = await fetch_consumption("IDF")
    return time.perf_counter() - t0

async def parallel():
    t0 = time.perf_counter()
    # asyncio.gather runs coroutines concurrently
    w, c = await asyncio.gather(
        fetch_weather("IDF"),
        fetch_consumption("IDF"),
    )
    return time.perf_counter() - t0

t_seq = asyncio.run(sequential())
t_par = asyncio.run(parallel())
print(f"Sequential: {t_seq*1000:.0f}ms")
print(f"Parallel  : {t_par*1000:.0f}ms  ← ~max(0.1, 0.15) = 0.15s")
print(f"Speedup   : {t_seq/t_par:.1f}x")

# ── asyncio.gather with many tasks ────────────────────────────
async def fetch_all_regions(regions: List[str]) -> List[dict]:
    tasks = [fetch_weather(r) for r in regions]
    return await asyncio.gather(*tasks)

regions = ["IDF", "PACA", "AURA", "HDF", "BFC"]
results = asyncio.run(fetch_all_regions(regions))
print(f"\nFetched {len(results)} regions concurrently")

In [ ]:
import time
import asyncio
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
import numpy as np

# ── ThreadPoolExecutor: run sync code in threads ──────────────
# Use when: calling blocking sync APIs inside async code
#           or parallelizing I/O-bound sync functions

def blocking_db_query(query_id: int) -> dict:
    "Simulates a blocking DB call (sync)"
    time.sleep(0.1)
    return {"query_id": query_id, "rows": 42}

# Run blocking function in thread pool from async context
async def async_db_query(query_id: int) -> dict:
    loop = asyncio.get_event_loop()
    with ThreadPoolExecutor(max_workers=4) as pool:
        return await loop.run_in_executor(pool, blocking_db_query, query_id)

async def fetch_many_queries():
    t0 = time.perf_counter()
    results = await asyncio.gather(*[async_db_query(i) for i in range(5)])
    print(f"5 queries in {(time.perf_counter()-t0)*1000:.0f}ms (parallel via threads)")
    return results

asyncio.run(fetch_many_queries())

# ── ProcessPoolExecutor: CPU-bound tasks ──────────────────────
def cpu_heavy(n: int) -> float:
    "Simulates CPU-intensive feature engineering"
    X = np.random.randn(n, 100)
    return float(np.linalg.svd(X, compute_uv=False).max())

# Sequential
t0 = time.perf_counter()
results_seq = [cpu_heavy(500) for _ in range(4)]
t_seq = time.perf_counter() - t0

# Parallel with ProcessPoolExecutor
t0 = time.perf_counter()
with ProcessPoolExecutor(max_workers=4) as pool:
    results_par = list(pool.map(cpu_heavy, [500, 500, 500, 500]))
t_par = time.perf_counter() - t0

print(f"\nCPU-bound sequential : {t_seq*1000:.0f}ms")
print(f"CPU-bound parallel   : {t_par*1000:.0f}ms")
print(f"Speedup              : {t_seq/t_par:.1f}x")

# KEY INSIGHT:
# async/await → I/O bound, single thread, cooperative (await yields control)
# ThreadPoolExecutor → I/O bound, multiple threads, good for sync blocking code
# ProcessPoolExecutor → CPU bound, multiple processes, bypasses GIL
# NEVER use async for CPU-heavy work → it doesn't help (no I/O to await)

In [ ]:
import asyncio
import time
from typing import List, Any

# ── Patterns: gather with error handling ──────────────────────
async def risky_fetch(source: str) -> dict:
    await asyncio.sleep(0.05)
    if source == "bad_source":
        raise ConnectionError(f"Cannot connect to {source}")
    return {"source": source, "ok": True}

async def gather_with_errors(sources: List[str]) -> List[Any]:
    "Gather results, collecting exceptions instead of raising"
    results = await asyncio.gather(
        *[risky_fetch(s) for s in sources],
        return_exceptions=True,   # exceptions become values in the list
    )
    successes = [r for r in results if not isinstance(r, Exception)]
    failures  = [r for r in results if isinstance(r, Exception)]
    print(f"Successes: {len(successes)} | Failures: {len(failures)}")
    for f in failures:
        print(f"  Error: {f}")
    return successes

sources = ["api_1", "bad_source", "api_2", "api_3"]
results = asyncio.run(gather_with_errors(sources))

# ── Pattern: timeout ──────────────────────────────────────────
async def slow_service(name: str, delay: float) -> str:
    await asyncio.sleep(delay)
    return f"{name} done"

async def with_timeout():
    try:
        result = await asyncio.wait_for(slow_service("model", 2.0), timeout=0.5)
        print(result)
    except asyncio.TimeoutError:
        print("Service timed out — falling back to default")
        return "default_prediction"

asyncio.run(with_timeout())

# ── Pattern: semaphore — limit concurrency ────────────────────
async def rate_limited_fetch(sem: asyncio.Semaphore, i: int) -> int:
    async with sem:
        await asyncio.sleep(0.05)
        return i * 2

async def controlled_concurrency():
    sem     = asyncio.Semaphore(3)   # max 3 concurrent tasks
    t0      = time.perf_counter()
    results = await asyncio.gather(*[rate_limited_fetch(sem, i) for i in range(10)])
    print(f"10 tasks (max 3 concurrent): {(time.perf_counter()-t0)*1000:.0f}ms")
    return results

asyncio.run(controlled_concurrency())

### 🏋️ Exercises 4.2

In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 1 — async pipeline: fetch + transform + score
# ════════════════════════════════════════════════════════
# Implement an async pipeline with 3 stages:
#
# async fetch_features(region: str) -> dict
#   simulates fetching raw features from a DB (sleep 0.1s)
#   returns {"region": region, "consumption": float, "temp": float}
#
# async transform_features(raw: dict) -> dict
#   applies log1p to consumption, rounds temp (sleep 0.02s)
#   returns {"region": str, "log_consumption": float, "temp": float}
#
# async score_region(region: str) -> dict
#   chains fetch → transform → scores with a fake model (sleep 0.03s)
#   score = log_consumption * 0.1 + temp * 0.01
#   returns {"region": str, "score": float}
#
# async score_all(regions: List[str]) -> List[dict]
#   runs score_region for all regions CONCURRENTLY
#   prints total time — should be ~max latency, not sum
#
# Test with 5 regions, show speedup vs sequential.

import asyncio, time, math
from typing import List

async def fetch_features(region: str) -> dict:
    pass  # complete here

async def transform_features(raw: dict) -> dict:
    pass  # complete here

async def score_region(region: str) -> dict:
    pass  # complete here

async def score_all(regions: List[str]) -> List[dict]:
    pass  # complete here

regions = ["IDF", "PACA", "AURA", "HDF", "BFC"]
results = asyncio.run(score_all(regions))
if results:
    for r in results:
        print(r)

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# async def fetch_features(region):
#     await asyncio.sleep(0.1)
#     import random; random.seed(hash(region) % 1000)
#     return {"region": region,
#             "consumption": random.uniform(80, 200),
#             "temp": random.uniform(5, 30)}
#
# async def transform_features(raw):
#     await asyncio.sleep(0.02)
#     return {"region": raw["region"],
#             "log_consumption": round(math.log1p(raw["consumption"]), 4),
#             "temp": round(raw["temp"], 2)}
#
# async def score_region(region):
#     raw   = await fetch_features(region)
#     feat  = await transform_features(raw)
#     await asyncio.sleep(0.03)
#     score = feat["log_consumption"] * 0.1 + feat["temp"] * 0.01
#     return {"region": region, "score": round(score, 4)}
#
# async def score_all(regions):
#     t0      = time.perf_counter()
#     results = await asyncio.gather(*[score_region(r) for r in regions])
#     elapsed = (time.perf_counter() - t0) * 1000
#     print(f"Scored {len(regions)} regions in {elapsed:.0f}ms "
#           f"(sequential would be ~{len(regions)*150:.0f}ms)")
#     return results

In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 2 — ThreadPoolExecutor: parallel batch scoring
# ════════════════════════════════════════════════════════
# You have a sync sklearn model. Write:
#
# score_batch_sequential(model, batches) -> List[np.ndarray]
#   loops over batches, calls model.predict_proba for each
#
# score_batch_parallel(model, batches, n_workers) -> List[np.ndarray]
#   uses ThreadPoolExecutor to score all batches concurrently
#   NOTE: sklearn's predict is thread-safe for inference
#
# Compare timing on 8 batches of 500 samples.
# Print speedup.

import numpy as np, time
from concurrent.futures import ThreadPoolExecutor
from typing import List
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=5000, n_features=20, random_state=42)
model = RandomForestClassifier(n_estimators=50, random_state=42).fit(X, y)

# Create batches
batches = [np.random.randn(500, 20) for _ in range(8)]

def score_batch_sequential(model, batches: List[np.ndarray]) -> List[np.ndarray]:
    pass  # complete here

def score_batch_parallel(model, batches: List[np.ndarray],
                          n_workers: int = 4) -> List[np.ndarray]:
    pass  # complete here

t0  = time.perf_counter()
r1  = score_batch_sequential(model, batches)
t_seq = time.perf_counter() - t0

t0  = time.perf_counter()
r2  = score_batch_parallel(model, batches, n_workers=4)
t_par = time.perf_counter() - t0

if r1 and r2:
    print(f"Sequential: {t_seq*1000:.0f}ms")
    print(f"Parallel  : {t_par*1000:.0f}ms")
    print(f"Speedup   : {t_seq/t_par:.1f}x")
    assert len(r1) == len(r2) == 8

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def score_batch_sequential(model, batches):
#     return [model.predict_proba(b) for b in batches]
#
# def score_batch_parallel(model, batches, n_workers=4):
#     with ThreadPoolExecutor(max_workers=n_workers) as pool:
#         futures = [pool.submit(model.predict_proba, b) for b in batches]
#         return [f.result() for f in futures]

---
## 4.3 Testing

### Lesson — pytest patterns for ML

In [ ]:
# pytest patterns — all runnable as-is via pytest or directly here

import pytest
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_breast_cancer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted

# ── Class under test ──────────────────────────────────────────
class OutlierClipper(BaseEstimator, TransformerMixin):
    def __init__(self, n_std: float = 3.0):
        self.n_std = n_std

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        self.mean_ = X.mean(axis=0)
        self.std_  = X.std(axis=0)
        return self

    def transform(self, X):
        check_is_fitted(self, ["mean_", "std_"])
        X = np.asarray(X, dtype=float)
        return np.clip(X,
                       self.mean_ - self.n_std * self.std_,
                       self.mean_ + self.n_std * self.std_)

# ── Fixtures ──────────────────────────────────────────────────
@pytest.fixture
def sample_data():
    np.random.seed(42)
    X = np.random.randn(100, 5)
    X[0, 0] = 100.0   # inject outlier
    return X

@pytest.fixture
def fitted_clipper(sample_data):
    return OutlierClipper(n_std=2.0).fit(sample_data)

@pytest.fixture
def breast_cancer():
    return load_breast_cancer(return_X_y=True)

# ── Tests ─────────────────────────────────────────────────────
def test_clipper_fit_stores_stats(sample_data):
    clipper = OutlierClipper().fit(sample_data)
    assert hasattr(clipper, "mean_")
    assert hasattr(clipper, "std_")
    assert clipper.mean_.shape == (5,)

def test_clipper_clips_outliers(sample_data, fitted_clipper):
    X_out = fitted_clipper.transform(sample_data)
    assert X_out[0, 0] < sample_data[0, 0]   # outlier clipped
    upper = fitted_clipper.mean_ + fitted_clipper.n_std * fitted_clipper.std_
    assert np.all(X_out <= upper + 1e-6)

def test_clipper_not_fitted_raises():
    clipper = OutlierClipper()
    with pytest.raises(Exception):   # NotFittedError
        clipper.transform(np.random.randn(5, 3))

def test_clipper_get_params():
    clipper = OutlierClipper(n_std=2.5)
    assert clipper.get_params() == {"n_std": 2.5}

@pytest.mark.parametrize("n_std", [1.0, 2.0, 3.0, 5.0])
def test_clipper_different_stds(sample_data, n_std):
    clipper = OutlierClipper(n_std=n_std).fit(sample_data)
    X_out   = clipper.transform(sample_data)
    upper   = clipper.mean_ + n_std * clipper.std_
    assert np.all(X_out <= upper + 1e-6)

def test_pipeline_with_clipper(breast_cancer):
    X, y = breast_cancer
    pipe = Pipeline([
        ("clip",  OutlierClipper(n_std=3.0)),
        ("scale", StandardScaler()),
        ("clf",   LogisticRegression(max_iter=1000)),
    ])
    pipe.fit(X[:80], y[:80])
    preds = pipe.predict(X[80:])
    assert preds.shape == (len(X) - 80,)
    assert set(preds).issubset({0, 1})

# Run them inline (normally you'd use: pytest test_module.py -v)
import traceback

tests = [
    test_clipper_fit_stores_stats,
    test_clipper_clips_outliers,
    test_clipper_not_fitted_raises,
    test_clipper_get_params,
]
data = np.random.randn(100, 5); data[0,0] = 100.0
clipper_2std = OutlierClipper(n_std=2.0).fit(data)
X_bc, y_bc = load_breast_cancer(return_X_y=True)

for fn in tests:
    try:
        if "sample_data" in fn.__code__.co_varnames and "fitted_clipper" in fn.__code__.co_varnames:
            fn(data, clipper_2std)
        elif "sample_data" in fn.__code__.co_varnames:
            fn(data)
        elif "breast_cancer" in fn.__code__.co_varnames:
            fn((X_bc, y_bc))
        else:
            fn()
        print(f"  PASS: {fn.__name__}")
    except Exception as e:
        print(f"  FAIL: {fn.__name__} — {e}")

### Course — TestClient + monkeypatch

In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field
from contextlib import asynccontextmanager
from unittest.mock import MagicMock, patch
from typing import Literal
import numpy as np, pytest

# ── App under test ────────────────────────────────────────────
class PredictionRequest(BaseModel):
    age: int = Field(ge=18, le=120)
    tenure_months: int = Field(ge=0)
    monthly_charges: float = Field(gt=0)
    contract_type: Literal["month-to-month", "one-year", "two-year"]

class PredictionResponse(BaseModel):
    churn_proba: float
    prediction: bool
    model_version: str

store = {}

@asynccontextmanager
async def lifespan(app: FastAPI):
    store["model"]   = MagicMock()
    store["model"].predict_proba = lambda X: np.array([[0.3, 0.7]])
    store["version"] = "v1.0"
    yield
    store.clear()

app = FastAPI(lifespan=lifespan)

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/predict", response_model=PredictionResponse)
def predict(req: PredictionRequest):
    proba = float(store["model"].predict_proba([[
        req.age, req.tenure_months, req.monthly_charges
    ]])[0][1])
    return PredictionResponse(
        churn_proba   = proba,
        prediction    = proba > 0.5,
        model_version = store["version"],
    )

# ── Tests with TestClient ─────────────────────────────────────
client = TestClient(app)

VALID_PAYLOAD = {
    "age": 35, "tenure_months": 24,
    "monthly_charges": 75.5, "contract_type": "one-year"
}

def test_health():
    r = client.get("/health")
    assert r.status_code == 200
    assert r.json()["status"] == "ok"

def test_predict_valid():
    r = client.post("/predict", json=VALID_PAYLOAD)
    assert r.status_code == 200
    body = r.json()
    assert "churn_proba" in body
    assert isinstance(body["prediction"], bool)
    assert body["model_version"] == "v1.0"
    assert 0.0 <= body["churn_proba"] <= 1.0

def test_predict_invalid_age():
    bad = {**VALID_PAYLOAD, "age": 200}
    r   = client.post("/predict", json=bad)
    assert r.status_code == 422

def test_predict_invalid_contract():
    bad = {**VALID_PAYLOAD, "contract_type": "weekly"}
    r   = client.post("/predict", json=bad)
    assert r.status_code == 422

def test_predict_missing_field():
    bad = {"age": 35, "tenure_months": 24}  # missing fields
    r   = client.post("/predict", json=bad)
    assert r.status_code == 422

# Run all tests
for test_fn in [test_health, test_predict_valid,
                test_predict_invalid_age, test_predict_invalid_contract,
                test_predict_missing_field]:
    try:
        test_fn()
        print(f"  PASS: {test_fn.__name__}")
    except AssertionError as e:
        print(f"  FAIL: {test_fn.__name__} — {e}")

# ── Mocking external dependencies ────────────────────────────
from unittest.mock import patch, MagicMock

# Mock joblib.load to avoid needing a real model file
with patch("joblib.load") as mock_load:
    mock_model = MagicMock()
    mock_model.predict_proba.return_value = np.array([[0.2, 0.8]])
    mock_load.return_value = mock_model

    import joblib
    model = joblib.load("fake_path.pkl")
    proba = model.predict_proba([[1, 2, 3]])[0][1]
    print(f"\nMocked model proba: {proba}")
    assert mock_load.called
    assert proba == 0.8
    print("Mock test passed")

### 🏋️ Exercises 4.3

In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 1 — Test suite for a custom transformer
# ════════════════════════════════════════════════════════
# Write a full pytest test suite for the RatioFeatures
# transformer from Module 3, Exercise 1.
# (reimplemented below for convenience)
#
# Required tests:
# 1. test_output_shape: n_features + len(pairs) columns
# 2. test_ratio_values: verify ratio is correctly computed
# 3. test_not_fitted_raises: transform before fit → exception
# 4. test_get_params: check pairs and epsilon are returned
# 5. test_in_pipeline: works end-to-end inside a Pipeline
# 6. parametrize on epsilon: [1e-8, 1e-5, 1e-3]

import numpy as np
import pytest
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.utils.validation import check_is_fitted
from sklearn.datasets import load_iris

class RatioFeatures(BaseEstimator, TransformerMixin):
    def __init__(self, pairs: list, epsilon: float = 1e-8):
        self.pairs   = pairs
        self.epsilon = epsilon

    def fit(self, X, y=None):
        self.n_features_in_ = np.asarray(X).shape[1]
        return self

    def transform(self, X):
        check_is_fitted(self, "n_features_in_")
        X = np.asarray(X, dtype=float)
        ratios = np.column_stack([
            X[:, i] / (X[:, j] + self.epsilon) for i, j in self.pairs
        ])
        return np.hstack([X, ratios])

# Write your tests here
def test_output_shape():
    pass  # complete

def test_ratio_values():
    pass  # complete

def test_not_fitted_raises():
    pass  # complete

def test_get_params():
    pass  # complete

def test_in_pipeline():
    pass  # complete

# Run
X_iris, y_iris = load_iris(return_X_y=True)

for fn in [test_output_shape, test_ratio_values,
           test_not_fitted_raises, test_get_params, test_in_pipeline]:
    try:
        fn()
        print(f"  PASS: {fn.__name__}")
    except Exception as e:
        if "pass" in str(e) or fn.__code__.co_consts == (None,):
            print(f"  TODO: {fn.__name__}")
        else:
            print(f"  FAIL: {fn.__name__} — {e}")

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# pairs = [(0, 1), (2, 3)]
#
# def test_output_shape():
#     X = np.random.randn(50, 4)
#     rf = RatioFeatures(pairs=pairs).fit(X)
#     out = rf.transform(X)
#     assert out.shape == (50, 4 + len(pairs))
#
# def test_ratio_values():
#     X = np.array([[2.0, 4.0, 6.0, 3.0]])
#     rf = RatioFeatures(pairs=[(0,1)], epsilon=0).fit(X)
#     out = rf.transform(X)
#     assert abs(out[0, -1] - 0.5) < 1e-6   # 2/4 = 0.5
#
# def test_not_fitted_raises():
#     rf = RatioFeatures(pairs=pairs)
#     with pytest.raises(Exception):
#         rf.transform(np.random.randn(5, 4))
#
# def test_get_params():
#     rf = RatioFeatures(pairs=pairs, epsilon=1e-5)
#     p  = rf.get_params()
#     assert p["pairs"] == pairs
#     assert p["epsilon"] == 1e-5
#
# def test_in_pipeline():
#     pipe = Pipeline([
#         ("rf",    RatioFeatures(pairs=pairs)),
#         ("scale", StandardScaler()),
#         ("clf",   LogisticRegression(max_iter=1000)),
#     ])
#     pipe.fit(X_iris, y_iris)
#     preds = pipe.predict(X_iris)
#     assert preds.shape == (150,)

In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 2 — TestClient: full API test suite
# ════════════════════════════════════════════════════════
# Write a test suite for the versioned API from Exercise 4.1.1
# (rebuild it here or copy your solution).
#
# Required tests (min 6):
# 1. test_health_returns_both_versions
# 2. test_v1_predict_valid_returns_200
# 3. test_v2_predict_lower_threshold
#    (same input → v2 predicts churn=True more often than v1)
# 4. test_predict_missing_field_returns_422
# 5. test_predict_invalid_contract_returns_422
# 6. test_model_list_has_correct_thresholds
# Bonus: parametrize test 2 over multiple valid payloads

from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field
from contextlib import asynccontextmanager
from typing import Literal

class PredictionRequest(BaseModel):
    age: int = Field(ge=18, le=120)
    tenure_months: int = Field(ge=0)
    monthly_charges: float = Field(gt=0)
    contract_type: Literal["month-to-month", "one-year", "two-year"]

class PredictionResponse(BaseModel):
    churn_proba: float
    prediction: bool
    model_version: str
    threshold_used: float

# Rebuild your app_ex1 from Exercise 4.1.1 here, then write tests

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# (rebuild app_ex1 as in 4.1 Exercise 1 solution, then:)
#
# client  = TestClient(app_ex1)
# PAYLOAD = {"age":30,"tenure_months":20,
#            "monthly_charges":60,"contract_type":"one-year"}
#
# def test_health_returns_both_versions():
#     r = client.get("/health")
#     assert r.status_code == 200
#     assert set(r.json()["models"]) == {"v1","v2"}
#
# def test_v1_predict_valid_returns_200():
#     r = client.post("/v1/predict", json=PAYLOAD)
#     assert r.status_code == 200
#     assert r.json()["model_version"] == "v1"
#
# def test_v2_predict_lower_threshold():
#     # high tenure → proba = 0.3+0.2=0.5
#     p = {"age":30,"tenure_months":20,"monthly_charges":60,"contract_type":"one-year"}
#     r1 = client.post("/v1/predict", json=p).json()
#     r2 = client.post("/v2/predict", json=p).json()
#     assert r2["threshold_used"] < r1["threshold_used"]
#
# def test_predict_missing_field_returns_422():
#     r = client.post("/v1/predict", json={"age":30})
#     assert r.status_code == 422
#
# def test_predict_invalid_contract_returns_422():
#     r = client.post("/v1/predict", json={**PAYLOAD, "contract_type":"weekly"})
#     assert r.status_code == 422
#
# def test_model_list_has_correct_thresholds():
#     r = client.get("/models")
#     models = {m["version"]: m["threshold"] for m in r.json()}
#     assert models["v1"] == 0.5
#     assert models["v2"] == 0.4

---
## 📋 Module 4 Recap

| Topic | Test |
|---|---|
| FastAPI routing | GET/POST with path params, query params, typed correctly |
| Pydantic I/O | request + response models with Field validation |
| lifespan | Load model once at startup, clean up on shutdown |
| HTTPException | 404, 422, 429, 500 — right status per error type |
| Depends | API key auth, pagination, shared model loader |
| async/await | Know when it helps (I/O) and when it doesn't (CPU) |
| asyncio.gather | Concurrent execution of coroutines |
| return_exceptions | Gather without crashing on partial failures |
| asyncio.wait_for | Add timeout to any coroutine |
| Semaphore | Cap concurrency inside gather |
| ThreadPoolExecutor | Run sync blocking code without blocking the event loop |
| ProcessPoolExecutor | CPU-bound parallel work |
| pytest fixture | Setup shared test state cleanly |
| @pytest.mark.parametrize | Test multiple inputs without repeating code |
| TestClient | Test FastAPI endpoints in-process, no server needed |
| MagicMock / patch | Replace external dependencies (joblib.load, DB calls) |
